## Isolated MicroVMs in Container Apps

Secure, isolated compute environments with sub-second startup.

Hardware-isolated microVM boundary — fully separated from host, platform, and other sandboxes
Snapshot-based suspend/resume preserving full memory and disk state across sessions
Per-sandbox network egress policy with deny-by-default posture for untrusted code

## What you can build

* **Traditional Apps.** Lift-and-shift workloads that need stateful compute, custom kernels, or per-tenant isolation without rewriting.
* **AI Apps & Agents.** Persistent, isolated workspaces that survive across task boundaries. Suspend between turns, resume with full context.
* **Code execution.** Run untrusted code in seconds with strong isolation. Capture state with snapshots, replay deterministically.
* **Dev environments.** Per-user compute that scales from zero to hundreds on demand and preserves state across sessions.
* **Many more…** CI runners, browser automation, data prep, reproducible experiments — anywhere a fast, isolated VM helps.

## Install the required Python packages

In [12]:
%pip install azure-containerapps-sandbox azure-identity azure-mgmt-resource azure-mgmt-authorization

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 13.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Get and set the required variables

In [28]:
subscription_id = ! az account show --query id -o tsv
subscription_id = subscription_id.n
print("Subscription ID : %s..." % subscription_id[:16])

principal_id = ! az ad signed-in-user show --query id -o tsv
principal_id = principal_id.n
print("Principal ID : %s..." % principal_id[:16])

resource_group = "rg-aca-sandbox"
sandbox_group = "aca-sandbox-group"
region = "swedencentral"

Subscription ID : dcef7009-6b94-43...
Principal ID : 8173c45d-035e-46...


## Create the resource group and sandbox group

In [30]:
import uuid
from azure.identity import DefaultAzureCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.containerapps.sandbox import (
    SandboxGroupManagementClient,
    SandboxGroupClient,
    endpoint_for_region,
)

credential = DefaultAzureCredential(
   exclude_cli_credential = False, 
   exclude_managed_identity_credential = True
)

# 1. Create resource group
resource_client = ResourceManagementClient(credential, subscription_id)
resource_client.resource_groups.create_or_update(resource_group, {"location": region})

# 2. Create sandbox group
mgmt = SandboxGroupManagementClient(
    credential, subscription_id=subscription_id, resource_group=resource_group,
)
mgmt.create_group(sandbox_group, location=region)

SandboxGroup(id='/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-sandbox/providers/Microsoft.App/sandboxGroups/aca-sandbox-group', name='aca-sandbox-group', location='swedencentral', tags={}, properties={'provisioningState': 'Succeeded', 'connections': [], 'allowedLocations': ['swedencentral'], 'managementEndpoint': 'https://management.swedencentral.azuredevcompute.io'}, identity=None)

## Create the RBAC role for the current user

In [ ]:
auth_client = AuthorizationManagementClient(credential, subscription_id)
scope = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
role_def = next(auth_client.role_definitions.list(
    scope, filter="roleName eq 'Container Apps SandboxGroup Data Owner'"
))
auth_client.role_assignments.create(scope, uuid.uuid4(), {
    "role_definition_id": role_def.id,
    "principal_id": principal_id,
    "principal_type": "User",
})

## Create the Sandbox

In [32]:
client = SandboxGroupClient(
    endpoint_for_region(region), credential,
    subscription_id=subscription_id,
    resource_group=resource_group,
    sandbox_group=sandbox_group,
)
sandbox = client.begin_create_sandbox(disk="ubuntu").result()

## Try out the Snandbox with a command

In [33]:
result = sandbox.exec("echo hello world && uname -a")
print(result.stdout)

hello world
Linux adc-sandbox 6.12.8+ #1 SMP Thu May 21 09:57:34 UTC 2026 x86_64 GNU/Linux



In [ ]:
# 6. Clean up
sandbox.delete()
mgmt.delete_group(sandbox_group)
client.close()
mgmt.close()